<a href="https://colab.research.google.com/github/Ayushsingh-d993/Generative-AI/blob/main/Rag_Application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Chat with your PDF***

In [ ]:
# WE are creating an API key of mistral AI, You are also using some different types of model and api like Open ai , google gemini etc.

MISTRALAI_API_KEY = "tr7VsothJ33TAyCPoXkTCrjqvpKwdR6Z"

In [ ]:
# Pass the API key from this program.

import os

from getpass import getpass

os.environ["MISTRAL_API_KEY"] = getpass("Enter your mistral api key: ")

Enter your mistral api key: ··········


In [ ]:
# # Install some requirement file or library to run your model.

!pip install -q pyngrok
!pip install chromadb
!pip install -q \
mistralai \
langchain \
langchain-community \
sentence-transformers \

pypdf \
python-dotenv \
!pip install langchain-mistralai
!pip install streamlit
!pip install streamlit pyngrok
!pip install langchain-mistralai


In [ ]:
# This is a main part of our program  where we are creating a file app.py  and write all the code and also creating a UI for this application using streamlit.

%%writefile app.py

import os
import tempfile
import streamlit as st

from langchain_mistralai import ChatMistralAI
from langchain_mistralai import MistralAIEmbeddings

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

from langchain_core.prompts import ChatPromptTemplate

# -------------------------
# Page Config
# -------------------------

st.set_page_config(
    page_title="PDF RAG Chat",
    page_icon="📚",
    layout="wide"
)

st.title("📚 Chat With Your PDF")

st.write("Upload a PDF and ask questions from it.")

# -------------------------
# Session State
# -------------------------

if "vectorstore" not in st.session_state:
    st.session_state.vectorstore = None

if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

# -------------------------
# Upload PDF
# -------------------------

uploaded_file = st.file_uploader(
    "Upload your PDF",
    type="pdf"
)

# -------------------------
# Process PDF
# -------------------------

if uploaded_file is not None:

    with st.spinner("Processing PDF..."):

        # Save PDF temporarily
        with tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".pdf"
        ) as tmp_file:

            tmp_file.write(uploaded_file.read())
            temp_pdf_path = tmp_file.name

        # Load PDF
        loader = PyPDFLoader(temp_pdf_path)
        docs = loader.load()

        # Split
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200
        )

        split_docs = text_splitter.split_documents(docs)

        # Embeddings
        embeddings = MistralAIEmbeddings(
            model="mistral-embed"
        )

        # Vector Store
        vectorstore = Chroma.from_documents(
            documents=split_docs,
            embedding=embeddings,
            persist_directory="./chromadb"
        )

        st.session_state.vectorstore = vectorstore

    st.success("PDF processed successfully!")

# -------------------------
# Chat
# -------------------------

if st.session_state.vectorstore is not None:

    retriever = st.session_state.vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": 4,
            "fetch_k": 10,
            "lambda_mult": 0.5
        }
    )

    # LLM
    model = ChatMistralAI(
        model="mistral-small-latest"
    )

    # Prompt
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
You are a helpful AI assistant.

Use ONLY the provided context to answer.

If answer is not found say:
'I could not find the answer in the document.'
"""
            ),
            (
                "human",
                """
Context:
{context}

Question:
{question}
"""
            )
        ]
    )

    # User Input
    user_query = st.chat_input(
        "Ask a question..."
    )

    if user_query:

        st.session_state.chat_history.append(
            ("user", user_query)
        )

        with st.spinner("Thinking..."):

            docs = retriever.invoke(user_query)

            context = "\n\n".join(
                [doc.page_content for doc in docs]
            )

            final_prompt = prompt.invoke({
                "context": context,
                "question": user_query
            })

            result = model.invoke(final_prompt)

            ai_response = result.content

        st.session_state.chat_history.append(
            ("assistant", ai_response)
        )

    # Display Chat
    for role, message in st.session_state.chat_history:

        with st.chat_message(role):
            st.write(message)

Overwriting app.py


In [ ]:
# We are running our file  in our background first.

!streamlit run app.py &>/content/logs.txt &

In [ ]:
# We are creating a authtoken of ngrok from official website of ngrok.

from pyngrok import ngrok

ngrok.set_auth_token("3E1xtXmuOQrqRnm9Frxdyo14UDw_639SLHjGaGQJGPZig8Zoi")

In [ ]:
# Finally connect with http for locally host our application. After run this file you will get the link of https.

from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://affix-carried-corrode.ngrok-free.dev" -> "http://localhost:8501"
